# Embeddings + FAISS Vector Store
In this notebook, we:
1. Load text chunks from **Step 1** (`outputs/chunks.jsonl`)
2. Generate embeddings using a Hugging Face model (`all-MiniLM-L6-v2`)
3. Store embeddings in a **FAISS** vector store
4. Save FAISS index for use in Step 3 (chatbot)

In [1]:
!pip install faiss-cpu sentence-transformers

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 236.1 kB/s eta 0:00:46
   - -------------------------------------- 0.5/11.3 MB 236.1 kB/s eta 0:00:46
   - -------------------------------------- 0.5/11.3 MB 236.1 kB/s eta 0

In [6]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path
base_path = Path(r"C:\Users\ADMIN\Documents\rag_chatbot")

chunks_path = base_path / "chunks.jsonl"
index_path = base_path / "faiss_index.bin"
metadata_path = base_path / "metadata.json"

In [5]:
# Load the text chunks from Step 1
documents = []
with open(chunks_path, 'r', encoding='utf-8') as f:
    for line in f:
        documents.append(json.loads(line))

print(f"Loaded {len(documents)} chunks")

Loaded 607 chunks


In [7]:
# Load Hugging Face embedding model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Extract texts
texts = [doc['text'] for doc in documents]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ADMIN\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Generate embeddings
embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f"Embeddings shape: {embeddings.shape}")

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embeddings shape: (607, 384)


In [9]:
# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"FAISS index contains {index.ntotal} vectors")

FAISS index contains 607 vectors


In [10]:
# Save FAISS index
faiss.write_index(index, str(index_path))

# Save metadata (map ids to texts)
metadata = {i: documents[i] for i in range(len(documents))}
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved FAISS index and metadata")

Saved FAISS index and metadata


In [11]:
# Example: search top 3 similar chunks
query = "What is the document about?"
query_vec = model.encode([query]).astype('float32')

distances, indices = index.search(query_vec, k=3)

for i, idx in enumerate(indices[0]):
    print(f"Result {i+1}: {metadata[idx]['text'][:200]}...\n")

Result 1: ature; here, we limit ourselves to a basic but still very general result
that applies to all of the examples we will consider. We will make one...

Result 2: 8.3 Splitting across Features 67
explanatory variables for any given outcome. For example, the obser-
vations may be a corpus of documents, and the features could include
all words and pairs of adjace...

Result 3: believe it should be.
The main contributions of this review can be summarized as follows:
(1) We provide a simple, cohesive discussion of the extensive
literature in a way that emphasizes and uniﬁes t...

